- **NASA POWER** ✅ (https://power.larc.nasa.gov)
  - Daily gridded weather data (temperature, precipitation, humidity, wind speed, solar radiation) for any lat/lon point globally
  - Covers 1981 to near real-time; no authentication required
  - Use case: weather and evapotranspiration inputs for irrigation models; temporal feature engineering

- **USDA NASS QuickStats** ✅ (https://quickstats.nass.usda.gov)
  - County-level crop statistics including yield, harvested area, production volume, and irrigated acreage
  - Annual surveys plus Census of Agriculture (every 5 years); free API key required
  - Use case: target variable (yield) for irrigation optimization models; benchmarking water use by region

- **USDA Cropland Data Layer (CDL)** ✅ (https://nassgeodata.gmu.edu/CropScape)
  - Annual 30m raster of crop types across the continental US, derived from satellite imagery
  - Available from 2008 onward; no authentication required; coordinates must be in CONUS Albers (EPSG:5070)
  - Use case: identifying which fields grow which crops; spatial stratification; combining with ET data at field level

- **OpenET** (https://openetdata.org)
  - Satellite-derived actual evapotranspiration at 30m field-scale resolution across the western US
  - Derived from Landsat imagery; free research account required; currently limited to western US
  - Use case: field-level water consumption estimates; more accurate than reference ET for actual crop water use

- **NASA SMAP** (https://smap.jpl.nasa.gov)
  - Satellite-derived soil moisture at ~9km resolution, updated every 2-3 days
  - Requires free NASA Earthdata account; data in HDF5 format; accessible via `earthaccess` Python library
  - Use case: near-real-time soil moisture input for irrigation scheduling models

- **SSURGO via Soil Data Access (SDA)** ✅ (https://sdmdataaccess.sc.egov.usda.gov)
  - Static soil properties (texture, drainage class, available water capacity, hydraulic conductivity) mapped at 1:24,000 scale
  - No authentication required; queried via SQL POST requests to SDMDataAccess.sc.egov.usda.gov
  - Use case: static soil covariates for irrigation and yield models; spatial stratification by soil type

- **USGS NWIS** ✅ (https://waterdata.usgs.gov/nwis)
  - Daily streamflow, gauge height, water temperature, and water quality data for thousands of gauging stations across the US
  - No authentication required; accessed via `dataretrieval` Python library
  - Use case: watershed-level water availability and drainage context for regional irrigation analysis

- **USDA LTAR / Ag Data Commons** ✅ (https://agdatacommons.nal.usda.gov)
  - Longitudinal soil, crop, and management data from the USDA Long-Term Agroecosystem Research network
  - Hosted on Figshare; no authentication required for public data; accessed via Figshare API at api.(figshare.com)[https://agdatacommons.nal.usda.gov/search?q=LTAR&itemTypes=3&categories=33725]
  - Use case: timestamped, management-aware soil and yield measurements; the primary source for dynamic soil properties that SSURGO cannot provide

- **3000 Rice Genomes Project (3K RGP)** (https://registry.opendata.aws/3kricegenome)
  - SNP genotype data for 3,024 rice varieties from 89 countries, sequenced at ~14x depth
  - Hosted on AWS S3 and NCBI SRA; no authentication required; ~15.4TB total
  - Use case: genotype side of rice GWAS; identifying SNPs associated with water use efficiency, nitrogen use efficiency, and other sustainability traits

- **IRRI SNP-Seek** (https://snp-seek.irri.org)
  - SNP genotype data and basic passport/phenotype data for the 3K rice panel, with analysis tools
  - No authentication required; accessible at snp-seek.irri.org; includes a downloadable 1M SNP GWAS dataset
  - Use case: more convenient access to 3K SNP data than raw SRA; includes some matched phenotype data for drought stress subsets

## NASA POWER API

In [12]:
"""
Hello World: NASA POWER API
============================
Daily weather data for a lat/lon point — no authentication required.

Parameters fetched (AG community):
  T2M               — mean air temperature at 2m (°C)
  T2M_MAX           — max air temperature at 2m (°C)
  T2M_MIN           — min air temperature at 2m (°C)
  PRECTOTCORR       — precipitation corrected (mm/day)
  RH2M              — relative humidity at 2m (%)
  WS2M              — wind speed at 2m (m/s)
  ALLSKY_SFC_SW_DWN — surface solar irradiance (MJ/m²/day)

ET₀ note:
  NASA POWER does not expose a pre-computed ET₀ parameter in the AG daily API.
  The variables above are the inputs to the FAO-56 Penman-Monteith equation,
  the standard method for computing reference ET₀. A Hargreaves-Samani
  temperature-based approximation is computed here for illustration.
  For production use, implement the full Penman-Monteith equation.

Full parameter list: https://power.larc.nasa.gov/docs/methodology/meteorology/
API docs:            https://power.larc.nasa.gov/docs/services/api/

Dependencies:
    pip install requests
"""

import requests


def fetch_nasa_power(lat=35.5, lon=-90.5, start="20220101", end="20220110"):
    """
    Fetch daily weather from NASA POWER for a lat/lon point and date range.

    Args:
        lat:   Latitude (decimal degrees)
        lon:   Longitude (decimal degrees)
        start: Start date as YYYYMMDD string
        end:   End date as YYYYMMDD string

    Returns:
        dict keyed by parameter name, each value a dict of {YYYYMMDD: float}
    """
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON",
    }

    print("── NASA POWER ──────────────────────────────────────────")
    print(f"Point:  ({lat}, {lon})")
    print(f"Period: {start} → {end}")
    print(f"URL:    {requests.Request('GET', url, params=params).prepare().url}\n")

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()

    # Response structure: data["properties"]["parameter"][param_name][YYYYMMDD]
    p = resp.json()["properties"]["parameter"]
    print(f"Parameters returned: {list(p.keys())}\n")
    return p


def hargreaves_et0(tmean, tmax, tmin, solar_mj_per_m2):
    """
    Hargreaves-Samani reference ET₀ estimate (mm/day).
    Simple temperature-based approximation; use Penman-Monteith for production.

    Args:
        tmean:          Mean daily temperature (°C)
        tmax:           Max daily temperature (°C)
        tmin:           Min daily temperature (°C)
        solar_mj_per_m2: Incoming solar radiation (MJ/m²/day)

    Returns:
        ET₀ in mm/day
    """
    # Convert solar radiation to mm/day equivalent (divide by latent heat of vaporization)
    ra = solar_mj_per_m2 / 0.408
    tr = max(tmax - tmin, 0)
    return 0.0023 * (tmean + 17.8) * (tr ** 0.5) * ra if ra > 0 else 0.0


def print_summary(p):
    """Print a formatted daily summary table with ET₀ estimates."""
    print(f"{'Date':<12} {'Tmean(°C)':>10} {'Precip(mm)':>11} {'RH(%)':>7} {'ET₀ est.(mm)':>13}")
    print("-" * 57)
    for date in sorted(p["T2M"].keys()):
        et0 = hargreaves_et0(
            tmean=p["T2M"][date],
            tmax=p["T2M_MAX"][date],
            tmin=p["T2M_MIN"][date],
            solar_mj_per_m2=p["ALLSKY_SFC_SW_DWN"][date],
        )
        print(
            f"{date:<12}"
            f"{p['T2M'][date]:>10.2f}"
            f"{p['PRECTOTCORR'][date]:>11.2f}"
            f"{p['RH2M'][date]:>7.1f}"
            f"{et0:>13.2f}"
        )


if __name__ == "__main__":
    data = fetch_nasa_power(
        lat=35.5,           # eastern Arkansas rice belt
        lon=-90.5,
        start="20220101",
        end="20220110",
    )
    print_summary(data)
    print("\n── Done ────────────────────────────────────────────────")

── NASA POWER ──────────────────────────────────────────
Point:  (35.5, -90.5)
Period: 20220101 → 20220110
URL:    https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M%2CT2M_MAX%2CT2M_MIN%2CPRECTOTCORR%2CRH2M%2CWS2M%2CALLSKY_SFC_SW_DWN&community=AG&longitude=-90.5&latitude=35.5&start=20220101&end=20220110&format=JSON

Parameters returned: ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'ALLSKY_SFC_SW_DWN']

Date          Tmean(°C)  Precip(mm)   RH(%)  ET₀ est.(mm)
---------------------------------------------------------
20220101         17.04      33.34   92.1         1.19
20220102         -0.24       0.33   73.4         0.83
20220103         -2.30       0.00   68.5         3.25
20220104         -0.05       0.03   83.2         3.16
20220105          4.11       0.07   73.7         4.77
20220106         -2.01       1.22   74.3         0.60
20220107         -4.25       0.00   68.9         1.91
20220108          2.70      21.58   88.8         2.24
20220109     

## NASS QuickStats API

In [13]:
"""
Hello World: USDA NASS QuickStats API
======================================
County-level crop statistics — yield, harvested area, irrigation.
Requires a free API key.
 
Registration (instant, free): https://quickstats.nass.usda.gov/api
 
Once registered, replace YOUR_NASS_API_KEY below with your key,
or set the environment variable NASS_API_KEY before running.
 
Useful stat categories:
  YIELD          — crop yield (e.g. BU / ACRE, CWT / ACRE)
  AREA HARVESTED — harvested acreage
  PRODUCTION     — total production volume
  AREA IRRIGATED — irrigated acreage (Census of Agriculture years only)
 
API docs: https://quickstats.nass.usda.gov/api
 
Dependencies:
    pip install requests
"""
 
import os
import requests
 
# Set your API key here or via the NASS_API_KEY environment variable
NASS_API_KEY = os.environ.get("NASS_API_KEY", "D010FDD0-1F8C-3CCA-8BFA-7345E97BD987")
 
 
def fetch_nass(
    commodity="RICE",
    state="ARKANSAS",
    year="2022",
    stat_cat="YIELD",
    agg_level="COUNTY",
    extra_params=None,
):
    """
    Fetch crop statistics from USDA NASS QuickStats.
 
    Args:
        commodity:    Crop name in uppercase (e.g. "RICE", "CORN", "SOYBEANS")
        state:        Full state name in uppercase (e.g. "ARKANSAS")
        year:         4-digit year as string
        stat_cat:     Statistic category (e.g. "YIELD", "AREA HARVESTED")
        agg_level:    Aggregation level: "COUNTY", "STATE", or "NATIONAL"
        extra_params: Optional dict of additional API parameters to merge in
 
    Returns:
        List of record dicts, or None if API key is not set.
    """
    if NASS_API_KEY == "YOUR_NASS_API_KEY":
        print("⚠  API key not set.")
        print("   Register at: https://quickstats.nass.usda.gov/api")
        print("   Then set NASS_API_KEY in this file or as an environment variable.")
        return None
 
    url = "https://quickstats.nass.usda.gov/api/api_GET/"
    params = {
        "key": NASS_API_KEY,
        "commodity_desc": commodity,
        "state_name": state,
        "year": year,
        "statisticcat_desc": stat_cat,
        "agg_level_desc": agg_level,
        "format": "JSON",
    }
    if extra_params:
        params.update(extra_params)
 
    print("── USDA NASS QuickStats ────────────────────────────────")
    print(f"Commodity: {commodity} | State: {state} | Year: {year}")
    print(f"Statistic: {stat_cat} | Level: {agg_level}")
    if extra_params:
        print(f"Extra:     {extra_params}")
    print()
 
    resp = requests.get(url, params=params, timeout=30)
 
    # NASS returns 400 when the query matches no data — inspect the body before raising
    if resp.status_code == 400:
        print(f"400 Bad Request — likely no data for this query combination.")
        try:
            print(f"API message: {resp.json()}")
        except Exception:
            print(f"Raw response: {resp.text[:300]}")
        return []
 
    resp.raise_for_status()
 
    records = resp.json().get("data", [])
    print(f"{len(records)} records returned.\n")
    return records
 
 
def print_summary(records, n=10):
    """Print a formatted table of the first n records."""
    if not records:
        print("No records to display.\n")
        return
    print(f"{'County':<25} {'Value':>12}  Unit")
    print("-" * 55)
    for rec in records[:n]:
        county = rec.get("county_name", "N/A")
        value  = rec.get("Value", "N/A")
        unit   = rec.get("unit_desc", "")
        print(f"{county:<25} {value:>12}  {unit}")
    if len(records) > n:
        print(f"  ... and {len(records) - n} more records")
    print()
 
 
if __name__ == "__main__":
 
    # Example 1: Rice yield by county in Arkansas, 2022 (annual survey)
    records = fetch_nass(
        commodity="RICE",
        state="ARKANSAS",
        year="2022",
        stat_cat="YIELD",
        agg_level="COUNTY",
    )
    print_summary(records)
 
    # Example 2: Rice harvested area from the Census of Agriculture, 2017.
    #
    # Notes on Census irrigation data:
    #   - Irrigated area is only collected in Census years: 2002, 2007, 2012, 2017, 2022
    #   - Must pass source_desc="CENSUS" — annual surveys do not include this field
    #   - County-level data for specific crops is sometimes suppressed by NASS
    #     for respondent confidentiality; if you get 0 records, try agg_level="STATE"
    #   - domain_desc="TOTAL" excludes sub-breakdowns by practice or method
    census = fetch_nass(
        commodity="RICE",
        state="ARKANSAS",
        year="2017",
        stat_cat="AREA HARVESTED",
        agg_level="COUNTY",
        extra_params={"source_desc": "CENSUS", "domain_desc": "TOTAL"},
    )
    print_summary(census)
 
    print("── Done ────────────────────────────────────────────────")

── USDA NASS QuickStats ────────────────────────────────
Commodity: RICE | State: ARKANSAS | Year: 2022
Statistic: YIELD | Level: COUNTY

23 records returned.

County                           Value  Unit
-------------------------------------------------------
CLAY                             7,610  LB / ACRE
GREENE                           7,270  LB / ACRE
JACKSON                          7,010  LB / ACRE
LAWRENCE                         7,320  LB / ACRE
MISSISSIPPI                      7,850  LB / ACRE
POINSETT                         6,890  LB / ACRE
ARKANSAS                         7,780  LB / ACRE
CRITTENDEN                       7,510  LB / ACRE
CROSS                            7,700  LB / ACRE
LEE                              7,240  LB / ACRE
  ... and 13 more records

── USDA NASS QuickStats ────────────────────────────────
Commodity: RICE | State: ARKANSAS | Year: 2017
Statistic: AREA HARVESTED | Level: COUNTY
Extra:     {'source_desc': 'CENSUS', 'domain_desc': 'TOTAL'}

140 

## USDA Cropland Data Layer (CDL)

In [17]:
"""
Hello World: USDA Cropland Data Layer (CDL)
============================================
Annual 30m raster of crop types across the continental US.
No authentication required.

IMPORTANT — Coordinate system:
  The CDL API expects bounding box coordinates in the CONUS Albers Equal Area
  projection (EPSG:5070), NOT WGS84 lat/lon. Passing lat/lon directly causes
  a 500 server error. This script reprojects your WGS84 bbox automatically
  using pyproj before sending the request.

Access pattern:
  1. Reproject WGS84 bbox → EPSG:5070 (Albers)
  2. GET request to CropScape with reprojected bbox
  3. Parse XML response to extract GeoTIFF download URL
  4. Download and read GeoTIFF with rasterio

Each pixel is an integer crop code. Common codes:
  0   Background / nodata     1   Corn
  2   Cotton                  3   Rice
  4   Sorghum                 5   Soybeans
  21  Barley                  24  Winter Wheat
  36  Alfalfa                 111 Open Water
  190 Woody Wetlands

Full code list:
  https://www.nass.usda.gov/Research_and_Science/Cropland/docs/cdl_codes.pdf

API docs: https://nassgeodata.gmu.edu/CropScape/

Dependencies:
    pip install requests rasterio numpy pyproj
"""

import requests
import numpy as np
import xml.etree.ElementTree as ET
from io import BytesIO
from pyproj import Transformer

try:
    import rasterio
except ImportError:
    raise ImportError(
        "rasterio is required for CDL raster parsing.\n"
        "Install with: pip install rasterio"
    )

# Common crop code lookup table
CDL_CODES = {
    0:   "Background",
    1:   "Corn",
    2:   "Cotton",
    3:   "Rice",
    4:   "Sorghum",
    5:   "Soybeans",
    6:   "Sunflower",
    21:  "Barley",
    22:  "Durum Wheat",
    23:  "Spring Wheat",
    24:  "Winter Wheat",
    36:  "Alfalfa",
    37:  "Other Hay/Non Alfalfa",
    111: "Open Water",
    121: "Developed/Open Space",
    141: "Deciduous Forest",
    190: "Woody Wetlands",
    195: "Herbaceous Wetlands",
}


def wgs84_to_albers(min_lon, min_lat, max_lon, max_lat):
    """
    Reproject a WGS84 bounding box to CONUS Albers (EPSG:5070).

    The CDL API requires coordinates in Albers projection — passing WGS84
    lat/lon directly will cause a 500 server error.

    Returns:
        (min_x, min_y, max_x, max_y) in Albers metres
    """
    transformer = Transformer.from_crs("epsg:4326", "epsg:5070", always_xy=True)
    min_x, min_y = transformer.transform(min_lon, min_lat)
    max_x, max_y = transformer.transform(max_lon, max_lat)
    return min_x, min_y, max_x, max_y


def fetch_cdl(year=2022, bbox_wgs84=(-91.2, 34.8, -91.0, 35.0)):
    """
    Download a CDL GeoTIFF for a WGS84 bounding box and return pixel array.

    Args:
        year:       Calendar year (CDL available from 2008 onward for most states)
        bbox_wgs84: (min_lon, min_lat, max_lon, max_lat) in WGS84 decimal degrees

    Returns:
        2D numpy array of crop codes, or None on failure.
    """
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84

    # Reproject to Albers — required by the CDL API
    min_x, min_y, max_x, max_y = wgs84_to_albers(min_lon, min_lat, max_lon, max_lat)
    albers_bbox = f"{min_x:.0f},{min_y:.0f},{max_x:.0f},{max_y:.0f}"

    url = (
        "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile"
        f"?year={year}&bbox={albers_bbox}"
    )

    print("── USDA Cropland Data Layer (CDL) ──────────────────────")
    print(f"Year:        {year}")
    print(f"BBox WGS84:  {bbox_wgs84}  (min_lon, min_lat, max_lon, max_lat)")
    print(f"BBox Albers: {albers_bbox}  (EPSG:5070 — required by CDL API)")
    print(f"URL:         {url}\n")

    # Step 1: request XML response containing GeoTIFF download URL
    resp = requests.get(url, timeout=60)

    if resp.status_code == 500:
        print("500 Server Error — common causes:")
        print("  - bbox outside continental US")
        print("  - CDL not available for this year/region")
        print(f"Raw response: {resp.text[:300]}")
        return None

    resp.raise_for_status()

    root = ET.fromstring(resp.content)

    # The download URL is in a <returnURL> element (namespace-agnostic search)
    url_elem = root.find(".//{*}returnURL") or root.find(".//returnURL")
    if url_elem is None or not url_elem.text:
        print("ERROR: Could not parse GeoTIFF URL from CDL response.")
        print(f"Raw response: {resp.text[:400]}")
        return None

    tiff_url = url_elem.text.strip()
    print(f"GeoTIFF URL: {tiff_url}\n")

    # Step 2: download GeoTIFF into memory
    tiff_resp = requests.get(tiff_url, timeout=60)
    tiff_resp.raise_for_status()

    # Step 3: open with rasterio and extract pixel array
    with rasterio.open(BytesIO(tiff_resp.content)) as src:
        data = src.read(1)   # single band — one crop code per pixel
        print(f"Raster shape: {data.shape[0]} rows × {data.shape[1]} cols")
        print(f"Resolution:   30m pixels")
        print(f"CRS:          {src.crs}\n")

    return data


def summarise(data, top_n=8):
    """Print a crop coverage summary table for a CDL pixel array."""
    codes, counts = np.unique(data, return_counts=True)
    total = counts.sum()

    print(f"{'Crop':<25} {'Code':>5} {'Pixels':>8} {'Coverage':>10}")
    print("-" * 52)
    for code, count in sorted(zip(codes, counts), key=lambda x: -x[1])[:top_n]:
        label = CDL_CODES.get(int(code), f"Code {code}")
        pct = count / total * 100
        print(f"{label:<25} {int(code):>5} {count:>8,} {pct:>9.1f}%")


if __name__ == "__main__":
    # Eastern Arkansas rice belt
    data = fetch_cdl(
        year=2022,
        bbox_wgs84=(-91.2, 34.8, -91.0, 35.0),
    )

    if data is not None:
        summarise(data)

    print("\n── Done ────────────────────────────────────────────────")

── USDA Cropland Data Layer (CDL) ──────────────────────
Year:        2022
BBox WGS84:  (-91.2, 34.8, -91.0, 35.0)  (min_lon, min_lat, max_lon, max_lat)
BBox Albers: 435427,1315425,452377,1338704  (EPSG:5070 — required by CDL API)
URL:         https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year=2022&bbox=435427,1315425,452377,1338704



/var/folders/xh/br51g1_j71sdmy2fqdss_blh0000gn/T/ipykernel_90745/1122336826.py:132: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  url_elem = root.find(".//{*}returnURL") or root.find(".//returnURL")


GeoTIFF URL: https://nassgeodata.gmu.edu/webservice/nass_data_cache/CDL_2022_clip_20260605010147_167041432.tif

Raster shape: 776 rows × 566 cols
Resolution:   30m pixels
CRS:          EPSG:5070

Crop                       Code   Pixels   Coverage
----------------------------------------------------
Soybeans                      5  194,725      44.3%
Rice                          3   75,312      17.1%
Woody Wetlands              190   62,529      14.2%
Corn                          1   47,973      10.9%
Code 61                      61   22,188       5.1%
Developed/Open Space        121    8,750       2.0%
Code 122                    122    6,761       1.5%
Cotton                        2    4,721       1.1%

── Done ────────────────────────────────────────────────


## USGS National Water Information System (NWIS)

In [21]:
import dataretrieval.waterdata as wd
import pandas as pd

# USGS NWIS Hello World
# Fetches daily streamflow data for a single gauging station in Iowa
#
# Site 05482000 = Raccoon River at Des Moines, IA
# Parameter code 00060 = discharge (streamflow) in cubic feet per second
# Time format is ISO 8601: "start/end" or "PnD" for last n days

df, metadata = wd.get_daily(
    monitoring_location_id="USGS-05482000",
    parameter_code="00060",
    time="2020-01-01T00:00:00Z/2020-12-31T00:00:00Z",
)

print("Columns:", df.columns.tolist())
print(df.head(10))
print(f"\nRows returned: {len(df)}")
print(f"\nMetadata: {metadata}")

# --- Notes ---
# monitoring_location_id format is always "USGS-{site_number}"
# To find site numbers for a state or region, use:
#   wd.get_monitoring_locations(state_name="Iowa", site_type="Stream")
#
# Common parameter codes:
#   00060 — discharge / streamflow (cfs)
#   00065 — gauge height (ft)
#   00010 — water temperature (C)
#   00400 — pH
#   00300 — dissolved oxygen (mg/L)
#
# Time can also be specified as a duration, e.g.:
#   time="P30D"  ->  last 30 days
#   time="P1Y"   ->  last year

Retrieving: daily · 1 page · 360 rows
No API key detected — register for higher rate limits at https://api.waterdata.usgs.gov/signup/


Columns: ['time_series_id', 'monitoring_location_id', 'parameter_code', 'statistic_id', 'time', 'value', 'unit_of_measure', 'approval_status', 'qualifier', 'last_modified', 'geometry', 'daily_id']
                     time_series_id monitoring_location_id parameter_code  \
0  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
1  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
2  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
3  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
4  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
5  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
6  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
7  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
8  8764e962882640ebad99d971b8bfe3d1          USGS-05482000          00060   
9  8764e962882640ebad99d971b8bfe3

## LTAR

In [28]:
import requests
import pandas as pd

# USDA LTAR / Ag Data Commons Hello World
#
# The Ag Data Commons is hosted on Figshare.
# Programmatic access goes through the main Figshare API at api.figshare.com,
# filtered to the USDA institution. No authentication required for public data.

BASE_URL = "https://api.figshare.com/v2"

def search_datasets(query, page_size=5):
    """Search Ag Data Commons datasets by keyword."""
    url = f"{BASE_URL}/articles/search"
    payload = {
        "search_for": query,
        "page_size": page_size,
    }
    response = requests.post(url, json=payload)
    response.raise_for_status()
    results = response.json()
    return pd.DataFrame([{
        "title": r["title"],
        "id":    r["id"],
        "doi":   r.get("doi", ""),
        "url":   r.get("url_public_html", ""),
    } for r in results])


def get_dataset_files(article_id):
    """Get downloadable files for a specific dataset."""
    url = f"{BASE_URL}/articles/{article_id}/files"
    response = requests.get(url)
    response.raise_for_status()
    files = response.json()
    return pd.DataFrame([{
        "name":         f["name"],
        "size_kb":      round(f["size"] / 1024, 1),
        "download_url": f["download_url"],
    } for f in files])


# Search for LTAR soil datasets
print("=== Searching for LTAR soil datasets ===")
results = search_datasets("LTAR soil", page_size=5)
print(results.to_string())

# Get files for the first result
if len(results) > 0:
    article_id = results.iloc[0]["id"]
    print(f"\n=== Files for: {results.iloc[0]['title']} ===")
    files = get_dataset_files(article_id)
    print(files.to_string())

    # Load the first CSV directly into a dataframe if one exists
    csv_files = files[files["name"].str.endswith(".csv")]
    if len(csv_files) > 0:
        csv_url = csv_files.iloc[0]["download_url"]
        print(f"\n=== Loading: {csv_files.iloc[0]['name']} ===")
        df = pd.read_csv(csv_url)
        print(df.head())

# --- Notes ---
# The search endpoint is a POST request with a JSON body (not a GET with params).
# institution=1577 filters results to USDA Ag Data Commons only.
#
# Useful search terms:
#   "LTAR soil"           — longitudinal soil measurements
#   "LTAR crop yield"     — yield data from LTAR network sites
#   "soil organic carbon" — soil carbon datasets
#   "nitrogen Iowa"       — fertilizer/nitrogen datasets for Iowa
#
# Once you have an article_id, get its files and load CSVs directly:
#   files = get_dataset_files(article_id)
#   df = pd.read_csv(files.iloc[0]["download_url"])
#
# Browse interactively at: https://agdatacommons.nal.usda.gov

=== Searching for LTAR soil datasets ===
                                                                                                                                                           title        id                            doi                                                                                                                                                                                                                      url
0                            Data from: Spatial and temporal drivers of landscape-level variation of aboveground net primary production in a semi-arid grassland  32065638  10.15482/USDA.ADC/32065638.v1                          https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Spatial_and_temporal_drivers_of_landscape-level_variation_of_aboveground_net_primary_production_in_a_semi-arid_grassland/32065638
1  Data from: Multivariate calibration of the Agricultural Policy / Environmental eXtender model for field scale simulati

In [26]:
import requests

# First, try a search without institution filter to confirm the API works
response = requests.post("https://api.figshare.com/v2/articles/search", json={
    "search_for": "LTAR soil",
    "page_size": 3
})
print("Status:", response.status_code)
print("Results:", len(response.json()))
print(response.json()[0]["title"] if response.json() else "empty")

Status: 200
Results: 3
Data from: Spatial and temporal drivers of landscape-level variation of aboveground net primary production in a semi-arid grassland
